In [ ]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

# Create an API client
import os
from anthropic import Anthropic

client = Anthropic()
model = os.getenv("CLAUDE_MODEL", "")

In [2]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

In [ ]:
import json
from statistics import mean 

def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert code reviewer. Evaluate this AI-generated solution.

    Task: {test_case["task"]}
    Solution: {output}

    Provide your evaluation as a structured JSON object with:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")

    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
    Please solve the following task:

    {test_case["task"]}
    """
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # Grade the output
    model_grade = grade_by_model(test_case, output) 
    score = model_grade["score"] 
    reasoning = model_grade["reasoning"]
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score, 
        "reasoning": reasoning
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    # Calculate an average score across all test cases
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}") 
    
    return results

In [7]:
with open("011_dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

# The evaluation returns a structured JSON array where each object represents one test case result
print(json.dumps(results, indent=2))

Average score: 5.833333333333333
[
  {
    "output": "# AWS S3 Bucket ARN Region Extractor\n\nHere's a Python function that extracts the AWS region from an S3 bucket ARN:\n\n```python\ndef extract_region_from_s3_arn(arn):\n    \"\"\"\n    Extracts the AWS region from an S3 bucket ARN.\n    \n    Args:\n        arn (str): S3 bucket ARN in the format 'arn:aws:s3:::bucket-name'\n                   or 'arn:aws:s3:region:account-id:bucket/bucket-name'\n    \n    Returns:\n        str: AWS region, or 'us-east-1' as default if region is not specified\n    \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        raise ValueError(\"ARN must be a non-empty string\")\n    \n    # Split ARN by colons\n    parts = arn.split(':')\n    \n    # Validate basic ARN structure\n    if len(parts) < 6 or parts[0] != 'arn' or parts[1] != 'aws':\n        raise ValueError(f\"Invalid ARN format: {arn}\")\n    \n    # S3 ARN format: arn:aws